# 도메인 전문가 에이전트: Phase 2-10 통합 프로젝트

> 기술의 가치는 배운 양이 아니라, 하나의 문제를 끝까지 풀어본 경험에서 나온다

Phase 2에서 토큰 하나를 이해하는 것으로 시작해서, Phase 10에서 자율적으로 판단하는 에이전트를 만든다.  
이 노트북은 그 전체 여정의 **최종 통합**이다.

---

## 학습 목표

| # | 목표 | 핵심 |
|---|------|------|
| 1 | **전체 기술 통합** | Phase 2~10의 모든 기술이 하나의 시스템으로 결합되는 구조를 이해한다 |
| 2 | **SQL 전문가 에이전트 구축** | 자연어 → SQL 생성 → 실행 → 검증 → 응답의 전체 파이프라인을 구현한다 |
| 3 | **커리큘럼 회고** | Phase 2~10 전체를 관통하는 파이프라인을 조감도로 정리한다 |

---

### 전제

- Ollama가 로컬에서 실행 중이라고 가정 (없으면 fallback 로직으로 동작)
- 모든 데이터는 in-memory sqlite3 — 설치 불필요
- Phase 4 파인튜닝 모델이 Ollama에 등록되어 있다고 가정 (없으면 규칙 기반 fallback)

In [ ]:
# === 환경 설정 ===
import json
import sqlite3
import re
import time
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import requests

# 한글 폰트
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# Ollama 설정
OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "llama3.2:latest"  # Phase 4 파인튜닝 모델 또는 기본 모델

def check_ollama():
    """Ollama 서버 연결 확인"""
    try:
        resp = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=3)
        models = [m['name'] for m in resp.json().get('models', [])]
        print(f"Ollama 연결 성공. 사용 가능 모델: {models[:5]}")
        return True
    except Exception:
        print("Ollama 미연결 — 규칙 기반 fallback 모드로 동작합니다.")
        return False

OLLAMA_AVAILABLE = check_ollama()

print("\n" + "=" * 60)
print("Phase 10-03: 도메인 전문가 에이전트 — 통합 프로젝트")
print("=" * 60)

---
## 1. 프로젝트 개요

이 에이전트는 Phase 2~10의 모든 기술이 하나의 시스템으로 결합된 최종 산출물이다.

### 기술 스택

| 계층 | 기술 | 역할 | 출처 Phase |
|------|------|------|------------|
| **추론 엔진** | Ollama | 파인튜닝 모델 로컬 서빙 | Phase 8 |
| **모델 (뇌)** | Phase 4 SFT 모델 | Text-to-SQL 특화 추론 | Phase 4 |
| **워크플로우** | LangGraph 패턴 | 멀티스텝 상태 관리 + 분기 | Phase 10 |
| **실행 환경** | sqlite3 | SQL 실행 엔진 | Phase 10 |
| **데이터** | Phase 3 파이프라인 | 학습 데이터 구성 | Phase 3 |
| **정렬** | DPO/GRPO | 모델 품질 정렬 | Phase 6 |
| **평가** | Phase 7 평가셋 | 성능 검증 프레임워크 | Phase 7 |
| **양자화** | GGUF (Q4_K_M) | 경량화 + 배포 | Phase 8 |

하나의 사용자 질문이 들어오면:  
`질문 이해 → 스키마 확인 → SQL 생성 → 검증 → 실행 → 결과 검증 → 응답 생성`  
이 전체 흐름을 자동으로 수행한다.

In [ ]:
# === 기술 스택 시각화 — 레이어 다이어그램 ===

fig, ax = plt.subplots(figsize=(12, 8))

# 4개 레이어 정의
layers = [
    {"name": "User Interface", "sub": "자연어 질문 입력 / 응답 출력",
     "color": "#339af0", "y": 3.0},
    {"name": "Workflow Engine", "sub": "LangGraph 패턴: 상태 관리 + 분기 + 재시도",
     "color": "#51cf66", "y": 2.0},
    {"name": "LLM Brain", "sub": "Ollama + Phase 4 SFT 모델 (Text-to-SQL)",
     "color": "#ffa94d", "y": 1.0},
    {"name": "Execution Environment", "sub": "sqlite3: SQL 실행 + 결과 반환",
     "color": "#ff6b6b", "y": 0.0},
]

for layer in layers:
    rect = plt.Rectangle((0.5, layer["y"]), 9, 0.8,
                          facecolor=layer["color"], edgecolor='black',
                          linewidth=2, alpha=0.85)
    ax.add_patch(rect)
    ax.text(5.0, layer["y"] + 0.5, layer["name"],
            ha='center', va='center', fontsize=14, fontweight='bold', color='white')
    ax.text(5.0, layer["y"] + 0.2, layer["sub"],
            ha='center', va='center', fontsize=10, color='white', alpha=0.9)

# 레이어 간 화살표
for i in range(len(layers) - 1):
    y_start = layers[i]["y"]
    y_end = layers[i + 1]["y"] + 0.8
    ax.annotate('', xy=(5.0, y_end + 0.02), xytext=(5.0, y_start - 0.02),
                arrowprops=dict(arrowstyle='<->', lw=2.5, color='#333333'))

# Phase 출처 표시 (오른쪽)
phase_labels = ["Phase 10", "Phase 10", "Phase 4,8", "Phase 10"]
for layer, phase in zip(layers, phase_labels):
    ax.text(10.0, layer["y"] + 0.4, phase,
            ha='left', va='center', fontsize=10, fontweight='bold',
            color='#555555', style='italic')

ax.set_xlim(0, 12)
ax.set_ylim(-0.5, 4.2)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('SQL 전문가 에이전트 — 기술 스택 레이어', fontsize=16, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

print("각 레이어는 Phase 2~10에서 구축한 기술의 결합이다.")
print("위에서 아래로: 질문 → 워크플로우 → LLM 추론 → DB 실행")

---
## 2. 데이터베이스 환경 구축

에이전트가 질의할 대상이 되는 **사내 데이터베이스**를 시뮬레이션한다.  
4개 테이블: `employees`, `departments`, `projects`, `assignments`

In [ ]:
# === sqlite3 in-memory DB 생성 ===

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# --- 테이블 생성 ---
cursor.executescript("""
CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL,
    location TEXT,
    budget REAL
);

CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    dept_id INTEGER,
    salary REAL,
    hire_date TEXT,
    manager_id INTEGER,
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id),
    FOREIGN KEY (manager_id) REFERENCES employees(emp_id)
);

CREATE TABLE projects (
    project_id INTEGER PRIMARY KEY,
    project_name TEXT NOT NULL,
    dept_id INTEGER,
    start_date TEXT,
    end_date TEXT,
    status TEXT,
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id)
);

CREATE TABLE assignments (
    assignment_id INTEGER PRIMARY KEY,
    emp_id INTEGER,
    project_id INTEGER,
    role TEXT,
    hours_worked REAL,
    FOREIGN KEY (emp_id) REFERENCES employees(emp_id),
    FOREIGN KEY (project_id) REFERENCES projects(project_id)
);
""")

# --- 샘플 데이터 삽입 ---
departments_data = [
    (1, '개발팀', '서울 본사', 500000000),
    (2, '마케팅팀', '서울 본사', 200000000),
    (3, '데이터팀', '판교', 350000000),
    (4, '인사팀', '서울 본사', 150000000),
    (5, '디자인팀', '판교', 180000000),
]
cursor.executemany("INSERT INTO departments VALUES (?,?,?,?)", departments_data)

employees_data = [
    (1, '김철수', 1, 8500, '2020-03-15', None),
    (2, '이영희', 1, 7200, '2021-06-01', 1),
    (3, '박지민', 1, 6800, '2022-01-10', 1),
    (4, '최수진', 2, 7000, '2020-08-20', None),
    (5, '정민호', 2, 5500, '2023-02-14', 4),
    (6, '한소영', 3, 9000, '2019-11-03', None),
    (7, '윤대현', 3, 7800, '2021-04-22', 6),
    (8, '강지훈', 3, 6500, '2022-09-01', 6),
    (9, '송미래', 4, 6000, '2021-07-15', None),
    (10, '임도윤', 4, 5200, '2023-05-30', 9),
    (11, '조현우', 1, 7500, '2020-12-01', 1),
    (12, '오세빈', 5, 7100, '2021-03-10', None),
    (13, '배수아', 5, 6300, '2022-06-15', 12),
    (14, '류태양', 3, 8200, '2020-05-20', 6),
    (15, '장하늘', 2, 5800, '2023-08-01', 4),
    (16, '김나윤', 1, 6900, '2022-11-20', 1),
    (17, '이준서', 3, 7400, '2021-09-05', 6),
    (18, '박서연', 5, 6600, '2022-04-12', 12),
    (19, '최도현', 2, 5300, '2023-10-01', 4),
    (20, '정예린', 4, 5700, '2022-08-25', 9),
]
cursor.executemany("INSERT INTO employees VALUES (?,?,?,?,?,?)", employees_data)

projects_data = [
    (1, 'AI 챗봇 개발', 1, '2024-01-01', '2024-06-30', '진행중'),
    (2, '브랜드 리뉴얼', 2, '2024-02-01', '2024-05-31', '완료'),
    (3, '데이터 파이프라인', 3, '2024-03-01', '2024-12-31', '진행중'),
    (4, '신규 채용 시스템', 4, '2024-04-01', '2024-08-31', '완료'),
    (5, 'UI/UX 개선', 5, '2024-05-01', '2024-11-30', '진행중'),
    (6, '추천 엔진', 3, '2024-01-15', '2024-09-30', '완료'),
    (7, '마케팅 자동화', 2, '2024-06-01', '2024-12-31', '진행중'),
    (8, '보안 감사 시스템', 1, '2024-03-15', '2024-07-31', '완료'),
]
cursor.executemany("INSERT INTO projects VALUES (?,?,?,?,?,?)", projects_data)

assignments_data = [
    (1, 1, 1, 'PM', 320),
    (2, 2, 1, '백엔드 개발', 480),
    (3, 3, 1, '프론트엔드 개발', 420),
    (4, 11, 1, 'AI 모델링', 500),
    (5, 4, 2, '기획', 200),
    (6, 5, 2, '콘텐츠', 180),
    (7, 15, 7, '마케팅 기획', 240),
    (8, 6, 3, 'PM', 350),
    (9, 7, 3, '데이터 엔지니어링', 420),
    (10, 8, 3, 'ETL 개발', 380),
    (11, 14, 6, 'ML 엔지니어링', 450),
    (12, 17, 6, '데이터 분석', 300),
    (13, 9, 4, 'PM', 160),
    (14, 10, 4, '시스템 개발', 200),
    (15, 12, 5, 'UI 디자인', 280),
    (16, 13, 5, 'UX 리서치', 220),
    (17, 18, 5, '프로토타이핑', 190),
    (18, 16, 8, '보안 개발', 350),
    (19, 3, 8, '코드 리뷰', 120),
    (20, 19, 7, '콘텐츠 제작', 160),
]
cursor.executemany("INSERT INTO assignments VALUES (?,?,?,?,?)", assignments_data)

conn.commit()

# --- 스키마 출력 ---
print("=" * 60)
print("데이터베이스 스키마")
print("=" * 60)

cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = [row[0] for row in cursor.fetchall()]

for table in tables:
    cursor.execute(f"PRAGMA table_info({table})")
    columns = cursor.fetchall()
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    
    print(f"\n[{table}] ({count}행)")
    print(f"  {'컬럼명':<20} {'타입':<12} {'PK':>3}  {'NULL허용':>8}")
    print(f"  {'-'*48}")
    for col in columns:
        cid, name, dtype, notnull, default, pk = col
        null_str = 'NOT NULL' if notnull else 'NULL OK'
        pk_str = 'PK' if pk else ''
        print(f"  {name:<20} {dtype:<12} {pk_str:>3}  {null_str:>8}")

print(f"\n총 {len(tables)}개 테이블, {sum(cursor.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in tables)}행")

---
## 3. SQL 전문가 에이전트 구축 (핵심 실습)

이 에이전트의 워크플로우:

```
질문 이해 → 스키마 확인 → SQL 생성 → SQL 검증 → SQL 실행 → 결과 검증 → 응답 생성
                              ↑                                    |
                              └──────── 실패 시 재시도 (max 3) ────┘
```

각 단계가 LangGraph의 **노드(Node)** 에 대응한다.  
실패 시 조건부 엣지(Conditional Edge)로 재시도 루프를 탄다.

In [ ]:
# === SQL 전문가 에이전트 클래스 ===

class SQLExpertAgent:
    """
    Phase 2-10 통합 SQL 전문가 에이전트
    
    워크플로우:
      질문 이해 → 스키마 확인 → SQL 생성 → 검증 → 실행 → 결과 검증 → 응답 생성
    """
    
    # 위험 키워드 (DML/DDL 차단)
    DANGEROUS_KEYWORDS = ['DROP', 'DELETE', 'TRUNCATE', 'ALTER', 'UPDATE', 'INSERT']
    
    # 테이블-키워드 매핑 (질문 이해 지원)
    TABLE_KEYWORDS = {
        'employees': ['직원', '사원', '이름', '급여', '연봉', '입사', '매니저', '상사'],
        'departments': ['부서', '팀', '위치', '예산', '지역'],
        'projects': ['프로젝트', '과제', '상태', '시작', '종료', '진행'],
        'assignments': ['배정', '참여', '역할', '시간', '투입', '인원'],
    }
    
    # 질문 패턴 → SQL 매핑 (규칙 기반 fallback)
    SQL_PATTERNS = {
        '이름.*모두|전체.*직원|직원.*목록': 'SELECT name FROM employees ORDER BY emp_id',
        '부서별.*평균.*급여|평균.*급여.*부서': 
            'SELECT d.dept_name, ROUND(AVG(e.salary), 0) as avg_salary '
            'FROM employees e JOIN departments d ON e.dept_id = d.dept_id '
            'GROUP BY d.dept_name ORDER BY avg_salary DESC',
        '평균보다.*높|급여.*높.*평균|부서.*평균.*초과':
            'SELECT e.name, e.salary, d.dept_name, '
            'ROUND(dept_avg.avg_sal, 0) as dept_avg_salary '
            'FROM employees e '
            'JOIN departments d ON e.dept_id = d.dept_id '
            'JOIN (SELECT dept_id, AVG(salary) as avg_sal FROM employees GROUP BY dept_id) dept_avg '
            'ON e.dept_id = dept_avg.dept_id '
            'WHERE e.salary > dept_avg.avg_sal '
            'ORDER BY e.salary DESC',
        '프로젝트.*인원|프로젝트.*참여.*시간|프로젝트별':
            'SELECT p.project_name, COUNT(a.emp_id) as member_count, '
            'SUM(a.hours_worked) as total_hours '
            'FROM projects p JOIN assignments a ON p.project_id = a.project_id '
            'GROUP BY p.project_name ORDER BY total_hours DESC',
        '매니저.*체인|상사.*따라|관리.*계층|보고.*라인':
            'WITH RECURSIVE manager_chain AS ('
            '  SELECT emp_id, name, manager_id, 0 as level '
            '  FROM employees WHERE manager_id IS NULL '
            '  UNION ALL '
            '  SELECT e.emp_id, e.name, e.manager_id, mc.level + 1 '
            '  FROM employees e '
            '  JOIN manager_chain mc ON e.manager_id = mc.emp_id '
            ') SELECT level, "  " || name as hierarchy, '
            'CASE WHEN manager_id IS NULL THEN "(최상위)" '
            'ELSE "→ 상사: " || (SELECT name FROM employees WHERE emp_id = manager_chain.manager_id) '
            'END as reports_to '
            'FROM manager_chain ORDER BY level, name',
    }
    
    def __init__(self, db_conn, model_name=MODEL_NAME, max_retries=3):
        self.conn = db_conn
        self.model_name = model_name
        self.max_retries = max_retries
        self.workflow_log = []  # 워크플로우 실행 로그
        self.performance_log = []  # 성능 기록
    
    def _log(self, step, message, status='info'):
        """워크플로우 단계 로깅"""
        icons = {'info': '[*]', 'success': '[+]', 'error': '[!]', 'warning': '[~]'}
        entry = {
            'step': step, 'message': message,
            'status': status, 'time': time.time()
        }
        self.workflow_log.append(entry)
        print(f"  {icons.get(status, '[*]')} {step}: {message}")
    
    # --- 단계 1: 질문 이해 ---
    def understand_question(self, question):
        """질문에서 의도와 관련 테이블을 추출"""
        self._log('질문 이해', f'입력: "{question}"')
        
        # 키워드 기반 테이블 식별
        related_tables = []
        for table, keywords in self.TABLE_KEYWORDS.items():
            if any(kw in question for kw in keywords):
                related_tables.append(table)
        
        # 아무 테이블도 안 잡히면 전체 포함
        if not related_tables:
            related_tables = list(self.TABLE_KEYWORDS.keys())
        
        self._log('질문 이해', f'관련 테이블: {related_tables}', 'success')
        return {'question': question, 'tables': related_tables}
    
    # --- 단계 2: 스키마 확인 ---
    def check_schema(self, context):
        """관련 테이블의 스키마를 조회"""
        schema_info = {}
        cursor = self.conn.cursor()
        
        for table in context['tables']:
            cursor.execute(f"PRAGMA table_info({table})")
            columns = cursor.fetchall()
            schema_info[table] = [
                {'name': col[1], 'type': col[2]} for col in columns
            ]
        
        self._log('스키마 확인', f'{len(schema_info)}개 테이블 스키마 로드', 'success')
        context['schema'] = schema_info
        return context
    
    # --- 단계 3: SQL 생성 ---
    def generate_sql(self, context):
        """SQL 쿼리 생성 (Ollama 또는 규칙 기반 fallback)"""
        question = context['question']
        
        # 방법 1: Ollama API 호출
        if OLLAMA_AVAILABLE:
            sql = self._generate_sql_ollama(question, context['schema'])
            if sql:
                self._log('SQL 생성', f'Ollama 모델 생성: {sql[:80]}...', 'success')
                context['sql'] = sql
                context['generation_method'] = 'ollama'
                return context
        
        # 방법 2: 규칙 기반 fallback
        sql = self._generate_sql_fallback(question)
        self._log('SQL 생성', f'규칙 기반 생성: {sql[:80]}...', 'success')
        context['sql'] = sql
        context['generation_method'] = 'rule-based'
        return context
    
    def _generate_sql_ollama(self, question, schema):
        """Ollama API로 SQL 생성"""
        schema_str = ""
        for table, cols in schema.items():
            col_str = ", ".join([f"{c['name']} {c['type']}" for c in cols])
            schema_str += f"{table}({col_str})\n"
        
        prompt = (
            f"다음 스키마에 대해 SQLite SQL 쿼리를 생성하세요.\n"
            f"스키마:\n{schema_str}\n"
            f"질문: {question}\n\n"
            f"SQL 쿼리만 출력하세요 (설명 없이):"
        )
        
        try:
            resp = requests.post(
                f"{OLLAMA_BASE_URL}/api/generate",
                json={"model": self.model_name, "prompt": prompt, "stream": False},
                timeout=30
            )
            result = resp.json().get('response', '')
            # SQL 추출
            sql_match = re.search(r'(SELECT.*?)(?:;|$)', result, re.DOTALL | re.IGNORECASE)
            if sql_match:
                return sql_match.group(1).strip()
        except Exception as e:
            self._log('SQL 생성', f'Ollama 호출 실패: {e}', 'warning')
        return None
    
    def _generate_sql_fallback(self, question):
        """규칙 기반 SQL 생성 (패턴 매칭)"""
        for pattern, sql in self.SQL_PATTERNS.items():
            if re.search(pattern, question):
                return sql
        # 기본 fallback
        return "SELECT * FROM employees LIMIT 10"
    
    # --- 단계 4: SQL 검증 ---
    def validate_sql(self, context):
        """SQL 구문 검증 + 위험 키워드 체크"""
        sql = context['sql'].upper()
        
        # 위험 키워드 체크
        found_dangerous = [kw for kw in self.DANGEROUS_KEYWORDS if kw in sql]
        if found_dangerous:
            self._log('SQL 검증', f'위험 키워드 감지: {found_dangerous}', 'error')
            context['validation'] = {'valid': False, 'reason': f'위험 키워드: {found_dangerous}'}
            return context
        
        # SELECT로 시작하는지 확인
        if not sql.strip().startswith('SELECT') and not sql.strip().startswith('WITH'):
            self._log('SQL 검증', 'SELECT/WITH 문이 아닙니다', 'error')
            context['validation'] = {'valid': False, 'reason': 'SELECT/WITH 문만 허용'}
            return context
        
        # EXPLAIN으로 구문 검증
        try:
            self.conn.execute(f"EXPLAIN {context['sql']}")
            self._log('SQL 검증', '구문 검증 통과', 'success')
            context['validation'] = {'valid': True}
        except sqlite3.Error as e:
            self._log('SQL 검증', f'구문 오류: {e}', 'error')
            context['validation'] = {'valid': False, 'reason': str(e)}
        
        return context
    
    # --- 단계 5: SQL 실행 ---
    def execute_sql(self, context):
        """sqlite3에서 SQL 실행"""
        if not context['validation']['valid']:
            context['result'] = None
            context['error'] = context['validation']['reason']
            return context
        
        try:
            start = time.time()
            cursor = self.conn.execute(context['sql'])
            columns = [desc[0] for desc in cursor.description] if cursor.description else []
            rows = cursor.fetchall()
            elapsed = time.time() - start
            
            context['result'] = {'columns': columns, 'rows': rows}
            context['execution_time'] = elapsed
            context['error'] = None
            self._log('SQL 실행', f'{len(rows)}행 반환 ({elapsed:.4f}s)', 'success')
        except sqlite3.Error as e:
            context['result'] = None
            context['error'] = str(e)
            self._log('SQL 실행', f'실행 오류: {e}', 'error')
        
        return context
    
    # --- 단계 6: 결과 검증 ---
    def verify_result(self, context):
        """결과 검증 (빈 결과 체크, 합리성 체크)"""
        if context['result'] is None:
            self._log('결과 검증', '결과 없음 — 재시도 필요', 'error')
            context['verified'] = False
            return context
        
        rows = context['result']['rows']
        
        # Empty check
        if len(rows) == 0:
            self._log('결과 검증', '결과가 비어있음 — 질의 재검토 필요', 'warning')
            context['verified'] = True  # 빈 결과도 유효할 수 있음
            return context
        
        # 합리성 체크: 결과가 과도하게 많은 경우
        if len(rows) > 1000:
            self._log('결과 검증', f'{len(rows)}행 — 결과가 과도, LIMIT 권장', 'warning')
        else:
            self._log('결과 검증', f'{len(rows)}행 — 정상', 'success')
        
        context['verified'] = True
        return context
    
    # --- 단계 7: 응답 생성 ---
    def format_response(self, context):
        """사용자 친화적 응답 생성"""
        if not context.get('verified') or context['result'] is None:
            return f"죄송합니다. '{context['question']}'에 대한 답변을 생성하지 못했습니다."
        
        columns = context['result']['columns']
        rows = context['result']['rows']
        
        # 테이블 형태 응답 생성
        response_lines = []
        response_lines.append(f"질문: {context['question']}")
        response_lines.append(f"생성된 SQL: {context['sql']}")
        response_lines.append(f"생성 방식: {context.get('generation_method', 'unknown')}")
        response_lines.append("")
        
        # 컬럼 헤더
        col_widths = []
        for i, col in enumerate(columns):
            max_w = len(str(col))
            for row in rows[:20]:  # 최대 20행 기준
                max_w = max(max_w, len(str(row[i])) if i < len(row) else 0)
            col_widths.append(min(max_w + 2, 30))
        
        header = " | ".join(str(col).ljust(col_widths[i]) for i, col in enumerate(columns))
        separator = "-+-".join("-" * w for w in col_widths)
        response_lines.append(header)
        response_lines.append(separator)
        
        for row in rows[:20]:
            line = " | ".join(
                str(row[i]).ljust(col_widths[i]) if i < len(row) else ""
                for i in range(len(columns))
            )
            response_lines.append(line)
        
        if len(rows) > 20:
            response_lines.append(f"... 외 {len(rows) - 20}행")
        
        response_lines.append(f"\n총 {len(rows)}행 반환")
        
        return "\n".join(response_lines)
    
    # --- 전체 워크플로우 실행 ---
    def run(self, question):
        """전체 워크플로우 실행 (with retry on error)"""
        self.workflow_log = []
        start_time = time.time()
        
        print(f"\n{'='*60}")
        print(f"질문: {question}")
        print(f"{'='*60}")
        
        for attempt in range(self.max_retries):
            if attempt > 0:
                print(f"\n  --- 재시도 {attempt}/{self.max_retries} ---")
            
            # 워크플로우 실행
            context = self.understand_question(question)
            context = self.check_schema(context)
            context = self.generate_sql(context)
            context = self.validate_sql(context)
            context = self.execute_sql(context)
            context = self.verify_result(context)
            
            # 성공 확인
            if context.get('verified') and context.get('result') is not None:
                total_time = time.time() - start_time
                response = self.format_response(context)
                
                # 성능 기록
                self.performance_log.append({
                    'question': question,
                    'success': True,
                    'attempts': attempt + 1,
                    'total_time': total_time,
                    'rows_returned': len(context['result']['rows']),
                    'method': context.get('generation_method', 'unknown'),
                })
                
                print(f"\n{response}")
                print(f"\n  총 소요 시간: {total_time:.4f}s (시도: {attempt + 1}회)")
                return context
        
        # 최대 재시도 초과
        total_time = time.time() - start_time
        self.performance_log.append({
            'question': question,
            'success': False,
            'attempts': self.max_retries,
            'total_time': total_time,
            'rows_returned': 0,
            'method': 'failed',
        })
        
        print(f"\n  최대 재시도 횟수({self.max_retries}회) 초과 — 실패")
        return context

print("SQLExpertAgent 클래스 정의 완료")
print("워크플로우: 질문이해 → 스키마확인 → SQL생성 → 검증 → 실행 → 결과검증 → 응답")

In [ ]:
# === 에이전트 테스트 — 5개 질문 (easy → hard) ===

agent = SQLExpertAgent(conn)

test_questions = [
    # Level 1: 단순 조회
    "직원 이름을 모두 알려줘",
    # Level 2: 집계 + JOIN
    "부서별 평균 급여를 알려줘",
    # Level 3: 서브쿼리
    "급여가 부서 평균보다 높은 직원을 찾아줘",
    # Level 4: 다중 JOIN + 집계
    "프로젝트별 참여 인원과 총 시간을 알려줘",
    # Level 5: 재귀 CTE
    "매니저 체인을 따라가서 보여줘",
]

difficulty_labels = ['Level 1: 단순 조회', 'Level 2: 집계+JOIN', 
                     'Level 3: 서브쿼리', 'Level 4: 다중JOIN+집계', 
                     'Level 5: 재귀CTE']

results = []
for i, (question, difficulty) in enumerate(zip(test_questions, difficulty_labels)):
    print(f"\n\n{'#'*60}")
    print(f"  테스트 {i+1}/5 — {difficulty}")
    print(f"{'#'*60}")
    result = agent.run(question)
    results.append(result)

print(f"\n\n{'='*60}")
print(f"전체 테스트 완료: {len(test_questions)}개 질문 처리")
print(f"{'='*60}")

---
## 4. 에러 핸들링: 자동 수정 루프

실전에서 LLM이 생성한 SQL은 **종종 실패**한다.  
에이전트의 핵심 가치는 실패를 감지하고 자동으로 재시도하는 **자기 수정 능력**에 있다.

```
생성 → 실패 → 에러 분석 → 재생성 → 성공
```

In [ ]:
# === 에러 시나리오 시뮬레이션 ===

class ErrorSimulationAgent(SQLExpertAgent):
    """의도적으로 첫 시도를 실패시켜 retry 흐름을 시연"""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._attempt_count = {}
    
    def generate_sql(self, context):
        question = context['question']
        attempt_key = question
        self._attempt_count[attempt_key] = self._attempt_count.get(attempt_key, 0) + 1
        
        if self._attempt_count[attempt_key] == 1:
            # 1차: 의도적으로 잘못된 SQL
            bad_sql = "SELECT name FROM non_existent_table"
            self._log('SQL 생성', f'1차 시도 (오류 포함): {bad_sql}', 'warning')
            context['sql'] = bad_sql
            context['generation_method'] = 'ollama (1차-오류)'
        elif self._attempt_count[attempt_key] == 2:
            # 2차: 구문은 맞지만 검증 실패할 SQL
            partial_sql = "SELECT name, salary FROM employees WHERE dept_id = 999"
            self._log('SQL 생성', f'2차 시도 (빈 결과): {partial_sql}', 'warning')
            context['sql'] = partial_sql
            context['generation_method'] = 'ollama (2차-개선)'
        else:
            # 3차: 올바른 SQL
            return super().generate_sql(context)
        
        return context

# 시뮬레이션 실행
print("에러 시나리오 시뮬레이션")
print("시나리오: LLM이 1차에서 잘못된 테이블명, 2차에서 빈 결과, 3차에서 성공\n")

error_agent = ErrorSimulationAgent(conn, max_retries=3)
error_result = error_agent.run("직원 이름을 모두 알려줘")

# --- retry 과정 시각화 ---
fig, ax = plt.subplots(figsize=(12, 4))

retry_steps = ['1차 시도\n(테이블명 오류)', '2차 시도\n(빈 결과)', '3차 시도\n(성공)']
retry_status = ['fail', 'partial', 'success']
colors = ['#ff6b6b', '#ffa94d', '#51cf66']

for i, (step, status, color) in enumerate(zip(retry_steps, retry_status, colors)):
    rect = plt.Rectangle((i * 3.5, 0), 2.8, 1.5,
                          facecolor=color, edgecolor='black', linewidth=2, alpha=0.85)
    ax.add_patch(rect)
    ax.text(i * 3.5 + 1.4, 0.75, step,
            ha='center', va='center', fontsize=10, fontweight='bold')
    
    if i < len(retry_steps) - 1:
        ax.annotate('', xy=(i * 3.5 + 3.0, 0.75), xytext=(i * 3.5 + 3.3, 0.75),
                    arrowprops=dict(arrowstyle='->', lw=2, color='#333'))
        ax.text(i * 3.5 + 3.15, 1.1, '에러 분석\n+ 재시도',
                ha='center', va='center', fontsize=8, color='#666')

ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.5, 2.0)
ax.axis('off')
ax.set_title('자동 수정 루프: 실패 → 에러 분석 → 재생성 → 성공', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 통계
print("\n시뮬레이션 통계:")
for log in error_agent.performance_log:
    print(f"  질문: {log['question']}")
    print(f"  성공: {log['success']} | 시도: {log['attempts']}회 | 시간: {log['total_time']:.4f}s")

---
## 5. 에이전트 성능 대시보드

에이전트가 처리한 질문들의 성능을 시각화한다.  
실전 배포 시 이 대시보드가 모니터링 시스템의 기반이 된다.

In [ ]:
# === 성능 시각화 4-panel 대시보드 ===

# 성능 데이터 집계
perf = agent.performance_log
if not perf:
    perf = [
        {'question': q, 'success': True, 'attempts': 1, 
         'total_time': np.random.uniform(0.001, 0.01), 
         'rows_returned': np.random.randint(3, 20), 'method': 'rule-based'}
        for q in test_questions
    ]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Panel 1: 질문 난이도별 성공률 ---
ax1 = axes[0, 0]
difficulties = difficulty_labels
success_rates = [100 if p['success'] else 0 for p in perf]
colors_bar = ['#51cf66' if s == 100 else '#ff6b6b' for s in success_rates]
bars = ax1.bar(range(len(difficulties)), success_rates, color=colors_bar, edgecolor='black')
ax1.set_xticks(range(len(difficulties)))
ax1.set_xticklabels([d.split(': ')[1] for d in difficulties], rotation=30, ha='right')
ax1.set_ylabel('성공률 (%)')
ax1.set_title('질문 난이도별 성공률')
ax1.set_ylim(0, 120)
for bar, rate in zip(bars, success_rates):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
             f'{rate}%', ha='center', fontweight='bold')
ax1.axhline(y=80, color='gray', linestyle='--', alpha=0.5, label='목표 80%')
ax1.legend()

# --- Panel 2: 평균 응답 시간 ---
ax2 = axes[0, 1]
times = [p['total_time'] * 1000 for p in perf]  # ms로 변환
short_names = [f'Q{i+1}' for i in range(len(perf))]
ax2.barh(short_names, times, color='#339af0', edgecolor='black')
ax2.set_xlabel('응답 시간 (ms)')
ax2.set_title('질문별 응답 시간')
for i, t in enumerate(times):
    ax2.text(t + max(times)*0.02, i, f'{t:.1f}ms', va='center', fontsize=9)

# --- Panel 3: retry 횟수 분포 ---
ax3 = axes[1, 0]
all_perf = perf + error_agent.performance_log
retry_counts = [p['attempts'] for p in all_perf]
retry_labels = ['1회 (즉시 성공)', '2회', '3회']
retry_values = [retry_counts.count(1), retry_counts.count(2), retry_counts.count(3)]
retry_colors = ['#51cf66', '#ffa94d', '#ff6b6b']
wedges, texts, autotexts = ax3.pie(retry_values, labels=retry_labels, colors=retry_colors,
                                    autopct='%1.0f%%', startangle=90,
                                    wedgeprops={'edgecolor': 'black', 'linewidth': 1.5})
ax3.set_title('재시도 횟수 분포')

# --- Panel 4: 워크플로우 단계별 소요 시간 (시뮬레이션) ---
ax4 = axes[1, 1]
workflow_steps = ['질문 이해', '스키마 확인', 'SQL 생성', 'SQL 검증', 'SQL 실행', '결과 검증', '응답 생성']
# 시뮬레이션 데이터 (실제로는 각 단계의 타이밍을 측정)
step_times = [0.5, 0.3, 8.0, 0.2, 1.5, 0.1, 0.4]  # ms 단위
step_colors = ['#339af0', '#339af0', '#ffa94d', '#51cf66', '#ff6b6b', '#51cf66', '#339af0']
bars4 = ax4.barh(workflow_steps[::-1], step_times[::-1], 
                  color=step_colors[::-1], edgecolor='black')
ax4.set_xlabel('소요 시간 (ms)')
ax4.set_title('워크플로우 단계별 소요 시간')
for bar, t in zip(bars4, step_times[::-1]):
    ax4.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             f'{t:.1f}ms', va='center', fontsize=9)

plt.suptitle('SQL 전문가 에이전트 — 성능 대시보드', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n핵심 인사이트:")
print("  - SQL 생성 단계가 전체 시간의 대부분을 차지 (LLM 추론 비용)")
print("  - 규칙 기반 fallback은 빠르지만, 복잡한 질문에는 LLM이 필수")
print("  - 자동 재시도로 최종 성공률을 끌어올릴 수 있다")

---
## 6. Phase 2-10 전체 파이프라인 회고

Phase 2에서 토큰 하나를 이해하는 것으로 시작해서, Phase 10에서 자율적으로 판단하는 에이전트를 만들었다.  
이 전체 여정을 하나의 조감도로 정리한다.

In [ ]:
# === Phase 2~10 전체 파이프라인 조감도 ===

fig, ax = plt.subplots(figsize=(16, 10))

# Phase 정의
phases = [
    {'name': 'Phase 2', 'title': 'Transformers 기초',
     'desc': '토큰화, 모델 구조, 추론', 'x': 1, 'y': 8, 'color': '#339af0'},
    {'name': 'Phase 3', 'title': '데이터 엔지니어링',
     'desc': 'Chat Template, 정제,\n합성 데이터', 'x': 5, 'y': 8, 'color': '#339af0'},
    {'name': 'Phase 4', 'title': 'SFT 파인튜닝',
     'desc': 'LoRA/QLoRA,\nText-to-SQL', 'x': 9, 'y': 8, 'color': '#51cf66'},
    {'name': 'Phase 5', 'title': '학습 안정성',
     'desc': 'Catastrophic Forgetting,\nW&B 모니터링', 'x': 13, 'y': 8, 'color': '#51cf66'},
    {'name': 'Phase 6', 'title': '선호도 정렬',
     'desc': 'DPO, GRPO', 'x': 1, 'y': 4.5, 'color': '#ffa94d'},
    {'name': 'Phase 7', 'title': '평가 시스템',
     'desc': '벤치마크,\n커스텀 평가셋', 'x': 5, 'y': 4.5, 'color': '#ffa94d'},
    {'name': 'Phase 8', 'title': '양자화 & 배포',
     'desc': 'GPTQ/AWQ/GGUF,\nvLLM, Ollama', 'x': 9, 'y': 4.5, 'color': '#ff6b6b'},
    {'name': 'Phase 9', 'title': 'RAG 시스템',
     'desc': '검색 증강 생성', 'x': 13, 'y': 4.5, 'color': '#ff6b6b'},
    {'name': 'Phase 10', 'title': '에이전트 시스템',
     'desc': 'LangGraph,\n도메인 전문가', 'x': 7, 'y': 1, 'color': '#784bd1'},
]

# 박스 그리기
box_w, box_h = 3.0, 2.0
for p in phases:
    rect = plt.Rectangle((p['x'] - box_w/2, p['y'] - box_h/2), box_w, box_h,
                          facecolor=p['color'], edgecolor='black',
                          linewidth=2, alpha=0.85, zorder=2)
    ax.add_patch(rect)
    ax.text(p['x'], p['y'] + 0.5, f"{p['name']}",
            ha='center', va='center', fontsize=11, fontweight='bold',
            color='white', zorder=3)
    ax.text(p['x'], p['y'] + 0.1, p['title'],
            ha='center', va='center', fontsize=9, color='white', zorder=3)
    ax.text(p['x'], p['y'] - 0.5, p['desc'],
            ha='center', va='center', fontsize=7, color='white', alpha=0.9, zorder=3)

# 화살표: 주요 흐름 (Phase 순서)
main_flow = [
    (0, 1), (1, 2), (2, 3),  # Phase 2→3→4→5 (상단)
    (2, 4),  # Phase 4→6 (SFT→DPO)
    (4, 5), (5, 6), (6, 7),  # Phase 6→7→8→9 (하단)
    (7, 8),  # Phase 9→10
    (6, 8),  # Phase 8→10 (Ollama)
    (2, 8),  # Phase 4→10 (파인튜닝 모델 = 에이전트의 뇌)
]

for src_idx, dst_idx in main_flow:
    src = phases[src_idx]
    dst = phases[dst_idx]
    ax.annotate('',
                xy=(dst['x'], dst['y'] + box_h/2 if dst['y'] < src['y'] else 
                    dst['x'] - box_w/2 if dst['x'] > src['x'] else dst['y'] + box_h/2),
                xytext=(src['x'], src['y'] - box_h/2 if dst['y'] < src['y'] else
                        src['x'] + box_w/2 if dst['x'] > src['x'] else src['y'] - box_h/2),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#555555'),
                zorder=1)

# 범례 (색상별 단계 구분)
legend_items = [
    ('기반 구축 (Phase 2-3)', '#339af0'),
    ('모델 학습 (Phase 4-5)', '#51cf66'),
    ('정렬+평가 (Phase 6-7)', '#ffa94d'),
    ('배포+확장 (Phase 8-9)', '#ff6b6b'),
    ('통합 (Phase 10)', '#784bd1'),
]
for i, (label, color) in enumerate(legend_items):
    ax.add_patch(plt.Rectangle((0.2, 0.5 - i * 0.6), 0.4, 0.4,
                                facecolor=color, edgecolor='black'))
    ax.text(0.8, 0.7 - i * 0.6, label, fontsize=8, va='center')

ax.set_xlim(-1.5, 16)
ax.set_ylim(-1, 10)
ax.axis('off')
ax.set_title('Phase 2-10 전체 파이프라인 조감도', fontsize=16, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

print("모든 화살표는 기술적 의존성을 나타낸다.")
print("Phase 10의 에이전트는 Phase 2~9의 모든 기술이 결합된 최종 산출물이다.")

In [ ]:
# === Phase별 핵심 산출물 요약 테이블 ===

phase_summary = [
    ('Phase 2', 'Transformers 기초', '토큰화/모델 로드 능력', '모든 Phase의 기반 기술'),
    ('Phase 3', '데이터 엔지니어링', '정제된 Chat 형식 학습 데이터', 'Phase 4 SFT의 입력 데이터'),
    ('Phase 4', 'SFT 파인튜닝', 'Text-to-SQL 파인튜닝 모델', 'Phase 6 정렬, Phase 10 에이전트의 뇌'),
    ('Phase 5', '학습 안정성', '안정적 학습 프로세스 + 모니터링', 'Phase 4, 6의 품질 보장'),
    ('Phase 6', '선호도 정렬', 'DPO/GRPO 정렬 모델', 'Phase 7 평가 대상'),
    ('Phase 7', '평가 시스템', '벤치마크 + 커스텀 평가 파이프라인', 'Phase 8 양자화 품질 검증'),
    ('Phase 8', '양자화 & 배포', 'GGUF 양자화 모델 + Ollama 서빙', 'Phase 10 에이전트의 추론 엔진'),
    ('Phase 9', 'RAG 시스템', '검색 증강 생성 파이프라인', '에이전트의 지식 확장 도구'),
    ('Phase 10', '에이전트 시스템', '도메인 전문가 에이전트', '최종 통합 산출물'),
]

print("=" * 90)
print("Phase 2-10 핵심 산출물 요약")
print("=" * 90)
print(f"{'Phase':<12} {'주제':<18} {'핵심 산출물':<32} {'다음 단계에서의 역할'}")
print("-" * 90)
for phase, topic, output, role in phase_summary:
    print(f"{phase:<12} {topic:<18} {output:<32} {role}")
print("=" * 90)

print("\n각 Phase는 독립적이지 않다.")
print("Phase 4의 파인튜닝 모델이 Phase 10 에이전트의 뇌가 되고,")
print("Phase 8의 Ollama가 Phase 10의 추론 엔진이 된다.")
print("전체가 하나의 파이프라인으로 연결된다.")

---
## 7. 다음 단계 제안: 실전 프로젝트

Phase 2~10의 기술 스택을 실전에 적용할 수 있는 3가지 프로젝트.

| # | 프로젝트 | 활용 기술 | 난이도 |
|---|---------|----------|--------|
| 1 | **사내 DB 자연어 인터페이스** | Phase 4 SFT + Phase 10 LangGraph + Streamlit | 중 |
| | 비개발자가 자연어로 사내 DB 조회. 스키마 자동 탐색 + SQL 생성 + 실행 + 시각화 | | |
| 2 | **도메인 문서 기반 Q&A 봇** | Phase 9 RAG + Phase 4 SFT + Phase 10 Agent | 중상 |
| | 사내 매뉴얼/규정집을 RAG로 검색하고, 파인튜닝 모델로 도메인 정확도를 높인 챗봇 | | |
| 3 | **자동 데이터 분석 에이전트** | Phase 10 LangGraph + Tool Use + Code Execution | 상 |
| | CSV 업로드 시 자동으로 EDA 수행, 인사이트 도출, 보고서 생성하는 멀티스텝 에이전트 | | |

세 프로젝트 모두 이 노트북의 SQL 전문가 에이전트를 **확장**하는 구조다.  
핵심은 동일하다: **LLM + 워크플로우 + 도구 + 검증 루프**.

---
## 8. 전체 커리큘럼 핵심 정리

| Phase | 핵심 역량 | 자기 진단 |
|-------|----------|----------|
| **Phase 2** | Transformer 모델을 로드하고 토큰화/추론을 수행할 수 있는가? | |
| **Phase 3** | Chat Template 형식으로 학습 데이터를 구성하고 정제할 수 있는가? | |
| **Phase 4** | LoRA/QLoRA로 모델을 파인튜닝하고 학습 곡선을 분석할 수 있는가? | |
| **Phase 5** | Catastrophic Forgetting을 진단하고 학습을 안정화할 수 있는가? | |
| **Phase 6** | DPO로 모델을 선호도 정렬하고 beta의 영향을 이해하는가? | |
| **Phase 7** | 벤치마크와 커스텀 평가셋으로 모델 성능을 정량 측정할 수 있는가? | |
| **Phase 8** | 모델을 양자화하고 Ollama/vLLM으로 배포할 수 있는가? | |
| **Phase 9** | RAG 파이프라인을 구축하여 외부 지식을 모델에 연결할 수 있는가? | |
| **Phase 10** | LangGraph로 멀티스텝 에이전트를 설계하고 파인튜닝 모델을 통합할 수 있는가? | |

**모든 항목에 '예'라고 답할 수 있다면, Phase 2~10 커리큘럼을 완주한 것이다.**

In [ ]:
# === Phase 10 + 전체 커리큘럼 최종 체크포인트 ===

print("=" * 60)
print("Phase 10 체크포인트")
print("=" * 60)

phase10_checks = [
    "LLM Agent의 4대 요소(LLM, 도구, 메모리, 계획)를 설명하고 구현할 수 있는가?",
    "파인튜닝 모델을 LangChain/Ollama로 연결하여 체인을 구성할 수 있는가?",
    "LangGraph의 StateGraph, Node, Conditional Edge를 이해하고 구현할 수 있는가?",
    "실패 자동 재시도 + Human-in-the-Loop을 포함한 멀티스텝 에이전트를 설계할 수 있는가?",
    "Phase 4 파인튜닝 모델을 에이전트의 뇌로 통합하여 도메인 전문가를 구축할 수 있는가?",
]

for i, check in enumerate(phase10_checks, 1):
    print(f"  [{i}] {check}")

print(f"\n\n{'='*60}")
print("전체 커리큘럼 최종 체크포인트")
print(f"{'='*60}")

curriculum_checks = [
    ("Phase 2", "Transformer 모델을 로드하고 토큰화/추론을 수행할 수 있는가?"),
    ("Phase 3", "Chat Template 형식으로 학습 데이터를 구성하고 정제할 수 있는가?"),
    ("Phase 4", "LoRA/QLoRA로 모델을 파인튜닝하고 학습 곡선을 분석할 수 있는가?"),
    ("Phase 5", "Catastrophic Forgetting을 진단하고 학습을 안정화할 수 있는가?"),
    ("Phase 6", "DPO로 모델을 선호도 정렬하고 beta의 영향을 이해하는가?"),
    ("Phase 7", "벤치마크와 커스텀 평가셋으로 모델 성능을 정량 측정할 수 있는가?"),
    ("Phase 8", "모델을 양자화하고 Ollama/vLLM으로 배포할 수 있는가?"),
    ("Phase 9", "RAG 파이프라인을 구축하여 외부 지식을 모델에 연결할 수 있는가?"),
    ("Phase 10", "LangGraph로 멀티스텝 에이전트를 설계하고 파인튜닝 모델을 통합할 수 있는가?"),
]

for phase, check in curriculum_checks:
    print(f"  [{phase}] {check}")

# DB 정리
conn.close()

print(f"\n\n{'='*60}")
print("  Phase 2에서 Attention을 이해하는 것으로 시작해서,")
print("  Phase 10에서 도메인 전문가 에이전트를 구축하기까지.")
print("")
print("  기술의 가치는 배운 양이 아니라,")
print("  하나의 문제를 끝까지 풀어본 경험에서 나온다.")
print(f"{'='*60}")